In [ ]:
!pip install groq

In [ ]:
!pip install -U langchain langchain-community faiss-cpu sentence-transformers pymupdf groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    F

In [ ]:
!pip install langchain-text-splitters

In [ ]:


from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import fitz  # PyMuPDF
import faiss
import numpy as np
from groq import Groq
import os
import re


In [ ]:
pdf_text = ""
doc = fitz.open("final_gt.pdf")
for page in doc:
      pdf_text += page.get_text()

In [ ]:
pdf_text = ""
doc = fitz.open("final_gt.pdf")
for page in doc:
      pdf_text += page.get_text()
sections = re.split(
    r'(--\s*(?:No Apparent|Mild|Moderate|Severe|Proliferative).*?Diabetic Retinopathy\s*--)',
    pdf_text
)


In [ ]:
disease_blocks = {}
current_disease = None

for block in sections:
    block = block.strip()

    if block.startswith("--"):
        current_disease = block.replace("-", "").strip()
        disease_blocks[current_disease] = ""
    elif current_disease:
        disease_blocks[current_disease] += " " + block


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=0
)

disease_chunks = {
    disease: text_splitter.split_text(text)
    for disease, text in disease_blocks.items()
}


In [ ]:
disease_blocks.keys()

dict_keys(['No Apparent Diabetic Retinopathy', 'Mild NonProliferative Diabetic Retinopathy', 'Moderate NonProliferative Diabetic Retinopathy', 'Severe NonProliferative Diabetic Retinopathy', 'Proliferative Diabetic Retinopathy'])

In [ ]:
emb = HuggingFaceEmbeddings(
    model_name="pritamdeka/biobert-mnli-snli-scinli-scitail-mednli-stsb"
)

faiss_indexes = {}
chunk_maps = {}

for disease, chunks in disease_chunks.items():
    embeddings = [emb.embed_query(c) for c in chunks]
    dim = len(embeddings[0])

    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings).astype("float32"))

    faiss_indexes[disease] = index
    chunk_maps[disease] = {i: c for i, c in enumerate(chunks)}


/tmp/ipython-input-778352893.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve_context(disease_name, query, k=5):
    index = faiss_indexes[disease_name]
    chunk_map = chunk_maps[disease_name]

    q_emb = emb.embed_query(query)
    D, I = index.search(np.array([q_emb], dtype="float32"), k)

    retrieved = [chunk_map[i] for i in I[0]]
    return " ".join(retrieved)



In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'-\s+', '', text)
    return text.strip()



In [ ]:
client = Groq(api_key="apikeyß")
def generate_report(retrieved_text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are operating in STRICT EXTRACTIVE MODE.\n"
                "You MUST NOT add, infer, assume, or introduce medical knowledge.\n"
                "You may ONLY rewrite or lightly clean sentences present in the text.\n\n"
                "If information is missing, write exactly:\n"
                "'Not stated in the retrieved text.'"
            )
        },
        {
            "role": "user",
            "content": f"""
TASK:
Transform the retrieved medical text into a structured report using ONLY the information explicitly present.

Retrieved Context (ONLY SOURCE OF TRUTH):
<<<
{retrieved_text}
>>>

FORMAT (MANDATORY):

1. Disease Overview (max 2 sentences)
2. Symptoms
3. Clinical Severity Assessment
4. Treatment Protocol & Medication Guidelines
5. Follow-up & Doctor Visit Schedule
6. Warning Signs (When to seek urgent help)

Generate the report now.
"""
        }
    ]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

    return response.choices[0].message.content


In [ ]:
predicted_disease = "Proliferative Diabetic Retinopathy"
context = retrieve_context(
    predicted_disease,
    "Disease Overview"
)
all_context = ""
all_context = all_context + "\n" + clean_text(context)


context = retrieve_context(
    predicted_disease,
    "give me any thing related to Symptoms"
)
all_context = all_context + "\n" + clean_text(context)

context = retrieve_context(
    predicted_disease,
    "give me any thing related to  Clinical Severity Assessment"
)
all_context = all_context + "\n" + clean_text(context)

context = retrieve_context(
    predicted_disease,
    "give me any thing related to  Treatment Protocol & Medication Guidelines"
)
all_context = all_context + "\n" + clean_text(context)

context = retrieve_context(
    predicted_disease,
    "give me any thing related to Follow-up & Doctor Visit Schedule"
)
all_context = all_context + "\n" + clean_text(context)



context = retrieve_context(
   predicted_disease,
    "give me any thing related to Warning Signs (When to seek urgent help)"
)
all_context = all_context + "\n" + clean_text(context)



print(generate_report(all_context))


1. Diabetic Retinopathy is a condition that requires urgent referral and close ophthalmic surveillance, particularly in cases of proliferative disease. The American Academy of Ophthalmology and other authoritative sources provide guidelines for the management of Diabetic Retinopathy.

2. Symptoms of Diabetic Retinopathy include sudden vision loss, persistent blurred vision, flashes of light, or new floaters, as well as a dark curtain or shadow in the visual field, eye pain or redness, and dense floaters suggesting vitreous hemorrhage.

3. Not stated in the retrieved text.

4. The standard treatment for Proliferative Diabetic Retinopathy (PDR) is Panretinal Photocoagulation (PRP), and Intravitreal anti-VEGF therapy is an effective alternative in appropriately selected patients, with medications such as ranibizumab, aflibercept, and bevacizumab being strongly recommended.

5. For mild non-proliferative diabetic retinopathy, ophthalmic follow-up is usually recommended every 9 to 12 months

In [ ]:
all_context.split("\n")[1]

'American Academy of Ophthalmology. Diabetic Retinopathy Preferred Practice Pattern, 2024. American Diabetes Association. Diabetic Retinopathy: A Position Statement. Diabetes Care, 2017. Wilkinson CP et al. International Clinical Diabetic Retinopathy Disease Severity Scale. Ophthalmology, 2003. Das T et al. Global Diabetic Retinopathy Screening Guidelines. Eye, 2021.'

In [ ]:
!pip install nltk


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(reference, candidate):
    smoothie = SmoothingFunction().method4
    ref_tokens = [reference.split()]
    cand_tokens = candidate.split()
    return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)


In [ ]:
gt_blocks = {}

In [ ]:
txt = """
Disease Overview: Mild NPDR is characterized by the presence of microaneurysms due to early
retinal microvascular damage.
Symptoms: Most patients are asymptomatic with preserved visual acuity.
Clinical Severity Assessment: Identified by isolated microaneurysms without significant ischemia.
Treatment Protocol & Medication Guidelines: Ocular treatment is usually not required; systemic
risk-factor control is emphasized.
Follow-up & Doctor Visit Schedule: Follow-up is typically recommended every 9–12 months.
Warning Signs: Sudden vision changes or new floaters require evaluation.
"""
gt_blocks ["Mild NonProliferative Diabetic Retinopathy"]= txt

txt = """
Disease Overview: Moderate NPDR reflects progression with intraretinal hemorrhages and
increasing ischemia.
Symptoms: Patients may experience blurred vision or contrast sensitivity changes.
Clinical Severity Assessment: Defined by hemorrhages, cotton wool spots, and venous changes.
Treatment Protocol & Medication Guidelines: Observation and systemic control are standard
unless complications develop.
Follow-up & Doctor Visit Schedule: Ophthalmic follow-up is typically every 3–6 months.
Warning Signs: New floaters, worsening vision, or distortion require prompt care.
"""
gt_blocks ["Moderate NonProliferative Diabetic Retinopathy"]=txt

txt = """
Disease Overview: Severe NPDR is marked by widespread capillary nonperfusion and high risk of
progression to PDR.
Symptoms: Visual distortion, dark spots, or intermittent vision loss may occur.
Clinical Severity Assessment: Defined by the 4-2-1 rule involving hemorrhages, venous beading,
and IRMA.
Treatment Protocol & Medication Guidelines: Close monitoring is required; early intervention
may be considered.
Follow-up & Doctor Visit Schedule: Follow-up is typically every 1–3 months.
Warning Signs: Sudden vision loss or symptoms of retinal detachment require urgent care.
"""

gt_blocks["Severe NonProliferative Diabetic Retinopathy"] = txt

txt = """
Disease Overview: PDR is the most advanced stage, characterized by pathological
neovascularization due to severe retinal ischemia.
Symptoms: Symptoms include sudden vision loss, dense floaters, shadows, or eye pain.
Clinical Severity Assessment: Defined by neovascularization or complications such as vitreous
hemorrhage.
Treatment Protocol & Medication Guidelines: Standard treatments include PRP, anti-VEGF
therapy, and surgery when indicated.
Follow-up & Doctor Visit Schedule: Close follow-up is required, often every 1–3 months during
active disease.
Warning Signs: Sudden vision loss, flashes, floaters, or eye pain require emergency evaluation.
"""
gt_blocks["Proliferative Diabetic Retinopathy"] = txt




In [ ]:
pred_blocks = {}
##copy from output of function after test each class and collect here to test
txt = """
1. Disease Overview:
Diabetic macular edema (DME) and progression are conditions that require treatment according to guideline-directed options. The presence of specific pathology, such as center-involving DME, is an indication for treatment, but not for uncomplicated mild NPDR.

2. Symptoms:
Sudden onset of floaters or a dramatic increase in floaters are symptoms that require immediate ophthalmic evaluation or emergency care.

3. Clinical Severity Assessment:
Not stated in the retrieved text, but it is implied that assessment is necessary to determine the presence of specific pathology and the severity of conditions like NPDR.

4. Treatment Protocol & Medication Guidelines:
Treatment options include antiVEGF for center-involving DME, laser, or referral for proliferative diabetic retinopathy (PDR), following guideline-directed options.

5. Follow-up & Doctor Visit Schedule:
Follow-up schedules include 6–12 months for moderate NPDR and 3–6 months for severe NPDR, with immediate referral if proliferative changes or center-involving DME appear.

6. Warning Signs (When to seek urgent help):
Patients should seek immediate ophthalmic evaluation or emergency care for sudden onset of floaters or a dramatic increase in floaters.

"""
pred_blocks["Mild NonProliferative Diabetic Retinopathy"] = txt

txt = """
1. Disease Overview:
Moderate NPDR is managed with observation and systemic control, and treatment is generally not required unless diabetic macular edema (DME) or progression occurs. Primary management is systemic metabolic control, including glycemia, blood pressure, and lipids.
2. Symptoms:
Not stated in the retrieved text, but patients are advised to seek care if symptoms suggest progression or complications.
3. Clinical Severity Assessment:
Moderate NPDR requires closer monitoring than mild NPDR.
4. Treatment Protocol & Medication Guidelines:
No routine panretinal photocoagulation (PRP) for moderate NPDR, and no routine anti-VEGF injections unless diabetic macular edema (DME) is present; ocular treatment is indicated only if complications develop.
5. Follow-up & Doctor Visit Schedule:
Typical follow-up interval is every 3–6 months.
6. Warning Signs (When to seek urgent help):
Warning signs include new or increasing blurred vision, new floaters, sudden reduction in vision, and distortion suggesting macular involvement.

"""
pred_blocks["Moderate NonProliferative Diabetic Retinopathy"]=txt

txt = """
1. Disease Overview:
Severe non-proliferative diabetic retinopathy (NPDR) has a high likelihood of progression and may warrant early treatment or close monitoring. Systemic metabolic control remains essential for managing the disease.
2. Symptoms:
Sudden visual symptoms may indicate vitreous hemorrhage, macular edema, or transition to proliferative disease, and include new vision loss, floaters, or flashes in diabetic patients.
3. Clinical Severity Assessment:
Not stated in the retrieved text, except that severe NPDR has a high likelihood of progression.
4. Treatment Protocol & Medication Guidelines:
Anti-VEGF therapy is not routine for severe NPDR alone, but may be used if diabetic macular edema (DME) is present or to reduce risk of progression in selected cases, and PRP can be considered in severe NPDR due to increased VEGF expression and ischemic burden.
5. Follow-up & Doctor Visit Schedule:
The typical follow-up interval is every 1–3 months, with frequent monitoring recommended for severe NPDR due to rapid progression risk, and patients with severe NPDR should be managed under hospital eye services with short-interval surveillance.
6. Warning Signs (When to seek urgent help):
 Patients should seek urgent eye examination for new vision loss, floaters, or flashes, as these symptoms may indicate vitreous hemorrhage, macular edema, or transition to proliferative disease.
"""

pred_blocks["Severe NonProliferative Diabetic Retinopathy"]=txt

txt = """
1. Diabetic Retinopathy is a condition that requires urgent referral and close ophthalmic surveillance, particularly in cases of proliferative disease. The American Academy of Ophthalmology and other authoritative sources provide guidelines for the management of Diabetic Retinopathy.

2. Symptoms of Diabetic Retinopathy include sudden vision loss, persistent blurred vision, flashes of light, or new floaters, as well as a dark curtain or shadow in the visual field, eye pain or redness, and dense floaters suggesting vitreous hemorrhage.

3. Not stated in the retrieved text.

4. The standard treatment for Proliferative Diabetic Retinopathy (PDR) is Panretinal Photocoagulation (PRP), and Intravitreal anti-VEGF therapy is an effective alternative in appropriately selected patients, with medications such as ranibizumab, aflibercept, and bevacizumab being strongly recommended.

5. For mild non-proliferative diabetic retinopathy, ophthalmic follow-up is usually recommended every 9 to 12 months, while for PDR, follow-up is typically every 1-3 months, especially during active disease or treatment, with more frequent visits during initiation of anti-VEGF therapy or post-PRP.

6. Warning signs that require immediate medical attention or urgent help include sudden vision loss, dense floaters, flashes of light, symptoms suggestive of retinal detachment, a dark curtain or shadow in the visual field, eye pain or redness, and rapid vision deterioration, difficulty reading, or central vision distortion.


"""
pred_blocks["Proliferative Diabetic Retinopathy"]=txt




In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def deduplicate_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(dict.fromkeys(sents))


In [ ]:
bleues= {}

gt_text = clean_text(deduplicate_sentences(gt_blocks["Proliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Proliferative Diabetic Retinopathy"])
bleues["Proliferative Diabetic Retinopathy"]= compute_bleu(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Severe NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Severe NonProliferative Diabetic Retinopathy"])
bleues["Severe NonProliferative Diabetic Retinopathy"] = compute_bleu(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Moderate NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Moderate NonProliferative Diabetic Retinopathy"])
bleues["Moderate NonProliferative Diabetic Retinopathy"] = compute_bleu(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Mild NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Mild NonProliferative Diabetic Retinopathy"])
bleues["Mild NonProliferative Diabetic Retinopathy"] = compute_bleu(gt_text, gen_text)




In [ ]:
bleues

{'Proliferative Diabetic Retinopathy': 0.04445435109045212,
 'Severe NonProliferative Diabetic Retinopathy': 0.06433396314328091,
 'Moderate NonProliferative Diabetic Retinopathy': 0.10204777092998345,
 'Mild NonProliferative Diabetic Retinopathy': 0.07287799183038302}

In [ ]:
!pip install bert-score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


In [ ]:
from bert_score import score

def compute_bertscore(reference, candidate, model="microsoft/deberta-xlarge-mnli"):
    P, R, F1 = score(
        [candidate],
        [reference],
        model_type=model,
        lang="en",
        verbose=False
    )
    return {
        "precision": P.item(),
        "recall": R.item(),
        "f1": F1.item()
    }


In [ ]:
berts= {}

gt_text = clean_text(deduplicate_sentences(gt_blocks["Proliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Proliferative Diabetic Retinopathy"])
berts["Proliferative Diabetic Retinopathy"]= compute_bertscore(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Severe NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Severe NonProliferative Diabetic Retinopathy"])
berts["Severe NonProliferative Diabetic Retinopathy"] = compute_bertscore(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Moderate NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Moderate NonProliferative Diabetic Retinopathy"])
berts["Moderate NonProliferative Diabetic Retinopathy"] = compute_bertscore(gt_text, gen_text)

gt_text = clean_text(deduplicate_sentences(gt_blocks["Mild NonProliferative Diabetic Retinopathy"]))
gen_text = clean_text(pred_blocks["Mild NonProliferative Diabetic Retinopathy"])
berts["Mild NonProliferative Diabetic Retinopathy"] = compute_bertscore(gt_text, gen_text)




tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

In [ ]:
berts

{'Proliferative Diabetic Retinopathy': {'precision': 0.5991169214248657,
  'recall': 0.6844541430473328,
  'f1': 0.6389487981796265},
 'Severe NonProliferative Diabetic Retinopathy': {'precision': 0.6007122993469238,
  'recall': 0.7161469459533691,
  'f1': 0.653370201587677},
 'Moderate NonProliferative Diabetic Retinopathy': {'precision': 0.639785885810852,
  'recall': 0.7636154890060425,
  'f1': 0.6962376236915588},
 'Mild NonProliferative Diabetic Retinopathy': {'precision': 0.5470819473266602,
  'recall': 0.6338309049606323,
  'f1': 0.5872701406478882}}